# 구조화된 출력 (Structured Output)

구조화된 출력을 사용하면 에이전트가 특정하고 예측 가능한 형식으로 데이터를 반환할 수 있습니다. 자연어 응답을 구문 분석하는 대신 애플리케이션에서 직접 사용할 수 있는 JSON 객체, Pydantic 모델 또는 데이터클래스 형태로 구조화된 데이터를 얻을 수 있습니다.

**구조화된 출력 전략:**

| 전략 | 설명 |
|:---|:---|
| **ProviderStrategy** | OpenAI, Anthropic, Grok 등 네이티브 구조화된 출력 지원 모델용 |
| **ToolStrategy** | 도구 호출을 통한 구조화된 출력 (대부분의 모델 지원) |
| **자동 선택** | 스키마 타입만 전달 시 모델에 따라 최적 전략 자동 선택 |

LangChain의 `create_agent`는 `response_format` 매개변수로 구조화된 출력을 설정하며, 결과는 에이전트 상태의 `structured_response` 키에 반환됩니다.

> 📖 **참고 문서**: [LangChain Structured Output](https://docs.langchain.com/oss/python/langchain/structured-output)

## 환경 설정

구조화된 출력 튜토리얼을 시작하기 전에 필요한 환경을 설정합니다. `dotenv`를 사용하여 API 키를 로드하고, `langchain_teddynote`의 로깅 기능을 활성화하여 LangSmith에서 실행 추적을 확인할 수 있도록 합니다.

LangSmith 추적을 활성화하면 에이전트의 구조화된 출력 생성 과정을 시각적으로 디버깅할 수 있어, 스키마 검증 오류나 재시도 과정을 파악하는 데 도움이 됩니다.

아래 코드는 환경 변수를 로드하고 LangSmith 프로젝트를 설정합니다.

In [4]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)

# LangSmith 추적 설정
# logging.langsmith("LangGraph-V1-Tutorial")

True

---

## Response Format

에이전트가 구조화된 데이터를 반환하는 방법을 `response_format` 매개변수로 제어합니다:

| 설정 | 설명 |
|:---|:---|
| **ToolStrategy[T]** | 도구 호출을 통한 구조화된 출력 |
| **ProviderStrategy[T]** | 제공자 네이티브 구조화된 출력 사용 |
| **type[T]** | 스키마 타입 직접 전달 - 모델에 따라 최적 전략 자동 선택 |
| **None** | 구조화된 출력 없음 |

스키마 타입이 직접 제공되면 LangChain이 자동으로 최적의 전략을 선택합니다:
- 네이티브 구조화된 출력 지원 모델(OpenAI, Anthropic, Grok)에는 `ProviderStrategy`
- 다른 모든 모델에는 `ToolStrategy`

구조화된 응답은 에이전트의 최종 상태의 `structured_response` 키에 반환됩니다.

---

## Provider Strategy

일부 모델 제공자(OpenAI, Anthropic, Grok 등)는 API를 통해 구조화된 출력을 네이티브로 지원합니다. 이 방법은 모델이 JSON 스키마를 직접 이해하고 준수하도록 강제하므로, 사용 가능한 경우 가장 신뢰할 수 있는 방법입니다.

스키마 타입을 `create_agent`의 `response_format`에 직접 전달하면, LangChain이 지원 모델에 대해 자동으로 `ProviderStrategy`를 사용합니다. 지원되지 않는 모델의 경우 자동으로 `ToolStrategy`로 폴백됩니다.

> 📖 **참고 문서**: [LangChain Models](https://docs.langchain.com/oss/python/langchain/models.md)

### Pydantic 모델

Pydantic 모델을 사용하면 필드에 대한 상세한 설명과 검증 규칙을 정의할 수 있습니다. `Field`의 `description`은 모델이 각 필드의 용도를 이해하는 데 도움이 되며, 이를 통해 더 정확한 구조화된 출력을 생성할 수 있습니다.

Pydantic은 타입 검증, 기본값 설정, 범위 제한(`ge`, `le`) 등 풍부한 검증 기능을 제공하므로, 복잡한 스키마 정의에 가장 적합합니다.

아래 코드는 Pydantic 모델로 연락처 정보를 추출하는 예시입니다.

In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """사람의 연락처 정보를 나타내는 스키마

    이름, 이메일, 전화번호를 구조화된 형태로 추출합니다.
    """

    name: str = Field(description="The name of the person")  # 이름
    email: str = Field(description="The email address of the person")  # 이메일 주소
    phone: str = Field(description="The phone number of the person")  # 전화번호


class Read(BaseModel):
    file_path: str
    start: int
    end: int



# 모델 초기화
# OpenAI 키 사용 시 gpt-5.4-mini, gpt-5.4 등으로 변경하세요.
model = init_chat_model("claude-sonnet-5-5")

# 에이전트 생성 - response_format에 스키마 타입 전달 시 ProviderStrategy 자동 선택
agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

# 구조화된 응답 확인
print(result["structured_response"])

/Users/jhj/Desktop/sds_ax_advanced_class/.venv/lib/python3.13/site-packages/langsmith/client.py:729: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8,id=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8; trace=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8,id=01a19ffc-b171-76d9-ab5f-b345b9522fa4; trace=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8,id=01a19ffc-b1c8-70ea-8522-022bf6945749


name='John Doe' email='john@example.com' phone='(555) 123-4567'


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8,id=01a19ffc-b1c8-70ea-8522-022bf6945749; trace=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8,id=01a19ffc-b171-76d9-ab5f-b345b9522fa4; trace=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8,id=01a19ffc-b14a-74e1-aabb-f67dcd52e9b8


### 데이터클래스

Python의 `@dataclass` 데코레이터를 사용하여 스키마를 정의할 수도 있습니다. 데이터클래스는 Pydantic보다 가볍고 Python 표준 라이브러리의 일부이므로 추가 의존성 없이 사용할 수 있습니다.

필드 설명은 주석(comment)으로 추가하며, 모델이 이를 참고하여 각 필드에 적절한 값을 채웁니다.

아래 코드는 데이터클래스로 연락처 정보를 추출하는 예시입니다.

In [3]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """사람의 연락처 정보를 나타내는 스키마"""

    name: str  # 이름 (The name of the person)
    email: str  # 이메일 주소 (The email address of the person)
    phone: str  # 전화번호 (The phone number of the person)


# 에이전트 생성 - 위에서 초기화한 model 재사용
agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,  # ProviderStrategy 자동 선택
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Jane Smith, jane@example.com, (555) 987-6543",
            }
        ]
    }
)

# 구조화된 응답 확인
print(result["structured_response"])

ContactInfo(name='Jane Smith', email='jane@example.com', phone='(555) 987-6543')


### TypedDict

`TypedDict`를 사용하면 딕셔너리 형태로 구조화된 출력을 받을 수 있습니다. 반환값이 Python 딕셔너리이므로 JSON 직렬화가 용이하고, API 응답이나 데이터베이스 저장에 바로 활용할 수 있습니다.

Pydantic이나 데이터클래스와 달리 인스턴스가 아닌 딕셔너리로 반환되므로, 후처리가 간단한 경우에 적합합니다.

아래 코드는 TypedDict로 연락처 정보를 추출하는 예시입니다.

In [4]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """사람의 연락처 정보를 나타내는 스키마"""

    name: str  # 이름 (The name of the person)
    email: str  # 이메일 주소 (The email address of the person)
    phone: str  # 전화번호 (The phone number of the person)


# 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo,  # ProviderStrategy 자동 선택
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: Teddy Lee, teddy@example.com, (555) 111-2222",
            }
        ]
    }
)

# 구조화된 응답 확인 - 딕셔너리 형태로 반환됨
print(result["structured_response"])

{'name': 'Teddy Lee', 'email': 'teddy@example.com', 'phone': '(555) 111-2222'}


---

## Tool Calling Strategy

네이티브 구조화된 출력을 지원하지 않는 모델의 경우, LangChain은 도구 호출(Tool Calling)을 사용하여 동일한 결과를 달성합니다. `ToolStrategy`는 도구 호출을 지원하는 대부분의 최신 모델에서 작동하므로 범용성이 높습니다.

`ToolStrategy`를 명시적으로 사용하면 네이티브 지원 여부와 관계없이 항상 도구 호출 방식을 사용합니다. 이는 일관된 동작이 필요하거나 특정 모델에서 네이티브 방식에 문제가 있을 때 유용합니다.

> 📖 **참고 문서**: [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)

### Pydantic 모델

`ToolStrategy`를 사용할 때도 Pydantic 모델의 필드 설명과 검증 규칙이 동일하게 적용됩니다. `Literal` 타입으로 허용값을 제한하고, `ge`(greater than or equal), `le`(less than or equal) 등의 검증자로 범위를 지정할 수 있습니다.

스키마 검증에 실패하면 에이전트가 자동으로 재시도하여 올바른 형식의 출력을 생성하도록 유도합니다.

아래 코드는 ToolStrategy로 제품 리뷰를 분석하는 예시입니다.

In [5]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """제품 리뷰 분석 결과를 나타내는 스키마"""

    rating: int | None = Field(
        description="The rating of the product", ge=1, le=5
    )  # 평점 (1-5)
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )  # 감정 분석 결과
    key_points: list[str] = Field(
        description="The key points of the review. Lowercase, 1-3 words each."
    )  # 핵심 포인트


# 에이전트 생성 - ToolStrategy 명시적 사용
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductReview),
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)

# 구조화된 응답 확인
print(result["structured_response"])

rating=5 sentiment='positive' key_points=['fast shipping', 'expensive', 'great product']


### Union 타입

`Union` 타입을 사용하여 여러 스키마 옵션을 제공할 수 있습니다. 모델은 입력 컨텍스트에 따라 가장 적절한 스키마를 자동으로 선택하므로, 다양한 유형의 입력을 하나의 에이전트로 처리할 수 있습니다.

예를 들어, 고객 피드백을 처리할 때 긍정적인 리뷰와 불만 사항을 서로 다른 스키마로 분류하여 각각에 맞는 후속 처리를 할 수 있습니다.

아래 코드는 리뷰와 불만을 구분하여 분석하는 예시입니다.

In [6]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """제품 리뷰 분석 결과를 나타내는 스키마
    
    긍정/부정 감정 분석과 핵심 포인트를 추출합니다.
    """

    rating: int | None = Field(
        description="The rating of the product", ge=1, le=5
    )  # 평점 (1-5)
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )  # 감정 분석 결과
    key_points: list[str] = Field(
        description="The key points of the review. Lowercase, 1-3 words each."
    )  # 핵심 포인트


class CustomerComplaint(BaseModel):
    """고객 불만 정보를 나타내는 스키마
    
    문제 유형, 심각도, 설명을 구조화된 형태로 추출합니다.
    """

    issue_type: Literal["product", "service", "shipping", "billing"] = Field(
        description="The type of issue"
    )  # 문제 유형
    severity: Literal["low", "medium", "high"] = Field(
        description="The severity of the complaint"
    )  # 심각도
    description: str = Field(
        description="Brief description of the complaint"
    )  # 불만 내용


# 에이전트 생성 - Union 타입으로 여러 스키마 지원
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[ProductReview, CustomerComplaint]),
)

# 리뷰 분석
result1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'",
            }
        ]
    }
)
print("Review:", result1["structured_response"])

# 불만 처리
result2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Customer complaint: Package arrived damaged and contents were broken",
            }
        ]
    }
)
print("Complaint:", result2["structured_response"])

Review: rating=5 sentiment='positive' key_points=['fast shipping', 'expensive', 'great product']
Complaint: issue_type='shipping' severity='high' description='Package arrived damaged and contents were broken'


### 커스텀 도구 메시지 콘텐츠

`tool_message_content` 매개변수를 사용하면 구조화된 출력이 생성될 때 대화 기록에 나타나는 메시지를 커스터마이징할 수 있습니다. 이 메시지는 후속 대화에서 컨텍스트를 제공하는 데 유용하며, 사용자에게 처리 상태를 알려주는 역할도 합니다.

기본적으로 도구 호출 결과는 JSON 형태로 메시지에 기록되지만, 커스텀 메시지를 설정하면 사람이 읽기 쉬운 형태로 표시됩니다. 이는 챗봇이나 대화형 인터페이스에서 사용자 경험을 개선하는 데 도움이 됩니다.

아래 코드는 회의 메모에서 액션 아이템을 추출하고, 커스텀 도구 메시지를 설정하는 예시입니다.

In [7]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """회의에서 추출된 액션 아이템을 나타내는 스키마
    
    담당자, 작업 내용, 우선순위를 구조화합니다.
    """

    task: str = Field(description="The specific task to be completed")  # 작업 내용
    assignee: str = Field(description="Person responsible for the task")  # 담당자
    priority: Literal["low", "medium", "high"] = Field(
        description="Priority level"
    )  # 우선순위


# 에이전트 생성 - 커스텀 도구 메시지 설정
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!",
    ),
)

# 에이전트 실행
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "From our meeting: Sarah needs to update the project timeline as soon as possible",
            }
        ]
    }
)

# 구조화된 응답 출력
print("=== Structured Response ===")
print(result["structured_response"])
# MeetingAction(task='Update the project timeline', assignee='Sarah', priority='high')

# 도구 메시지 콘텐츠 출력
print("=== Tool Message Content ===")
print(result["messages"][-1].content)
# Action item captured and added to meeting notes!

=== Structured Response ===
task='Update the project timeline' assignee='Sarah' priority='high'
=== Tool Message Content ===
Action item captured and added to meeting notes!


---

## 오류 처리

모델은 도구 호출을 통해 구조화된 출력을 생성할 때 스키마와 일치하지 않는 값을 반환할 수 있습니다. 예를 들어, 1-5 범위로 지정된 rating에 10을 반환하거나, 필수 필드를 누락하는 경우가 있습니다. LangChain은 이러한 오류를 자동으로 감지하고 처리하는 지능형 재시도 메커니즘을 제공합니다.

`handle_errors` 매개변수로 오류 처리 방법을 세밀하게 제어할 수 있으며, 기본값은 `True`로 모든 검증 오류를 자동으로 처리합니다. 오류가 발생하면 에이전트는 모델에게 오류 내용을 피드백하고 재시도를 요청하여, 최종적으로 스키마에 맞는 출력을 얻도록 합니다.

이 기능은 프로덕션 환경에서 안정적인 구조화된 출력을 보장하는 데 필수적입니다.

### 스키마 검증 오류

구조화된 출력이 예상 스키마와 일치하지 않으면 에이전트는 오류 피드백을 제공하고 모델에게 재시도를 요청합니다. 예를 들어, rating이 1-5 범위인데 사용자가 "10/10"이라고 입력하면, 모델이 이를 10으로 파싱하려다 검증 오류가 발생하고 자동으로 5로 수정됩니다.

이 재시도 메커니즘 덕분에 실제 사용자 입력이 예상 범위를 벗어나더라도 안정적으로 처리할 수 있습니다. LangSmith에서 실행 추적을 확인하면 재시도 과정을 시각적으로 볼 수 있습니다.

아래 코드는 스키마 검증 오류가 발생했을 때 자동으로 수정되는 예시입니다.

In [8]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    """제품 평점 정보를 나타내는 스키마
    
    평점은 1-5 범위로 제한되며, 리뷰 코멘트를 포함합니다.
    """

    rating: int | None = Field(
        description="Rating from 1-5", ge=1, le=5
    )  # 평점 (1-5 범위)
    comment: str = Field(description="Review comment")  # 리뷰 코멘트


# 에이전트 생성 - 기본값 handle_errors=True로 자동 재시도 활성화
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductRating),  # 기본값: handle_errors=True
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up.",
)

# 10/10 입력 - 범위를 벗어나므로 자동으로 5로 수정됨
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]}
)

# 구조화된 응답 확인
print(result["structured_response"])
# ProductRating(rating=5, comment='Amazing product')
# 모델이 자동으로 수정하여 10을 5로 변경

rating=5 comment='Amazing product, 10/10!'


### 오류 처리 전략

`handle_errors` 매개변수를 사용하여 오류 처리 방법을 다양하게 커스터마이징할 수 있습니다. 요구사항에 따라 자동 재시도, 예외 발생, 커스텀 핸들러 등을 선택할 수 있습니다.

| 설정 | 설명 |
|:---|:---|
| `True` | 모든 오류를 자동으로 처리하고 재시도 (기본값) |
| `False` | 오류 발생 시 예외 발생 |
| 문자열 | 커스텀 오류 메시지로 재시도 |
| 예외 클래스 | 특정 예외만 처리 |
| 콜러블 | 커스텀 오류 핸들러 함수 사용 |

아래에서 각 전략의 사용 예시를 살펴봅니다.

### 커스텀 오류 메시지

문자열을 `handle_errors`에 전달하면 해당 메시지로 모델에게 재시도를 요청합니다. 기본 오류 메시지 대신 더 구체적인 안내를 제공할 수 있으므로, 모델이 오류를 더 잘 이해하고 올바른 형식으로 수정할 수 있습니다.

특히 복잡한 스키마나 특정 도메인의 규칙이 있는 경우, 커스텀 메시지로 명확한 지시를 제공하면 재시도 성공률이 높아집니다.

아래 코드는 커스텀 오류 메시지를 설정하는 예시입니다.

In [9]:
from langchain.agents.structured_output import ToolStrategy

# 커스텀 오류 메시지로 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating,
        # 오류 발생 시 이 메시지로 재시도 요청
        handle_errors="Please provide a valid rating between 1-5 and include a comment.",
    ),
)

### 특정 예외만 처리

예외 클래스를 `handle_errors`에 전달하면 해당 예외만 처리하고 다른 예외는 그대로 발생합니다. 이는 예상된 검증 오류만 재시도하고, 예상치 못한 시스템 오류는 즉시 감지하고 싶을 때 유용합니다.

예를 들어, `ValueError`만 처리하면 범위 검증 오류는 재시도하지만, 네트워크 오류나 타입 불일치 같은 다른 오류는 예외로 발생시킵니다.

아래 코드는 특정 예외만 처리하는 예시입니다.

In [10]:
# ValueError만 처리하는 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating,
        handle_errors=ValueError,  # ValueError만 재시도, 다른 예외는 그대로 발생
    ),
)

### 여러 예외 유형 처리

튜플로 여러 예외 클래스를 `handle_errors`에 전달하면 해당 예외들을 모두 처리합니다. 이는 여러 종류의 검증 오류를 한 번에 처리하고 싶을 때 유용합니다.

일반적으로 `ValueError`와 `TypeError`를 함께 처리하면 대부분의 스키마 검증 오류를 커버할 수 있습니다.

아래 코드는 여러 예외 유형을 처리하는 예시입니다.

In [11]:
# 여러 예외 유형을 처리하는 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating,
        # ValueError 및 TypeError 모두 재시도
        handle_errors=(ValueError, TypeError),
    ),
)

### 커스텀 오류 핸들러 함수

함수(콜러블)를 `handle_errors`에 전달하면 오류 발생 시 해당 함수가 호출되어 커스텀 오류 메시지를 생성합니다. 이 방법은 오류 유형에 따라 다른 메시지를 반환하거나, 오류 정보를 로깅하는 등 복잡한 처리가 필요할 때 유용합니다.

LangChain은 `StructuredOutputValidationError`(스키마 검증 실패)와 `MultipleStructuredOutputsError`(여러 출력 반환) 등의 구체적인 예외 클래스를 제공합니다. 이를 활용하면 오류 유형별로 세밀한 처리가 가능합니다.

아래 코드는 커스텀 오류 핸들러 함수를 사용하는 예시입니다.

In [12]:
from langchain.agents.structured_output import (
    StructuredOutputValidationError,
    MultipleStructuredOutputsError,
)


def custom_error_handler(error: Exception) -> str:
    """오류 유형에 따른 커스텀 오류 메시지 생성
    
    Args:
        error: 발생한 예외 객체
        
    Returns:
        모델에게 전달할 오류 메시지
    """
    if isinstance(error, StructuredOutputValidationError):
        # 스키마 검증 오류
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        # 여러 구조화된 출력이 반환된 경우
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        # 기타 오류
        return f"Error: {str(error)}"


# 커스텀 오류 핸들러로 에이전트 생성
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating, handle_errors=custom_error_handler
    ),
)

### 오류 처리 비활성화

`handle_errors=False`를 설정하면 오류 발생 시 예외가 그대로 발생합니다. 이는 개발 중에 오류를 디버깅하거나, 커스텀 오류 처리 로직을 직접 구현하고 싶을 때 유용합니다.

프로덕션 환경에서는 일반적으로 `handle_errors=True`(기본값)를 사용하여 안정적인 동작을 보장하지만, 개발 및 테스트 환경에서는 비활성화하여 문제를 빠르게 파악할 수 있습니다.

아래 코드는 오류 처리를 비활성화하는 예시입니다.

In [13]:
# 오류 처리 비활성화 - 모든 오류가 예외로 발생
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=ProductRating, handle_errors=False  # 오류 발생 시 예외 발생
    ),
)

---

## 종합 예제

지금까지 학습한 `Union` 타입과 오류 처리를 결합한 실용적인 예제입니다. 하나의 에이전트가 책과 영화 추천을 모두 처리하며, 입력 내용에 따라 적절한 스키마를 자동으로 선택합니다.

`handle_errors=True`로 설정하여 오류 발생 시 자동으로 재시도하므로, 프로덕션 환경에서도 안정적으로 동작합니다. 이 패턴은 다양한 유형의 입력을 처리해야 하는 챗봇이나 AI 어시스턴트에 적합합니다.

아래 코드는 책과 영화 추천을 처리하는 에이전트 예시입니다.

In [14]:
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class BookRecommendation(BaseModel):
    """책 추천 정보를 나타내는 스키마
    
    제목, 저자, 장르, 평점, 요약을 구조화합니다.
    """

    title: str = Field(description="Book title")  # 책 제목
    author: str = Field(description="Author name")  # 저자명
    genre: Literal["fiction", "non-fiction", "science", "history", "biography"] = Field(
        description="Book genre"
    )  # 장르
    rating: int = Field(description="Rating from 1-5", ge=1, le=5)  # 평점 (1-5)
    summary: str = Field(description="Brief summary of the book")  # 책 요약


class MovieRecommendation(BaseModel):
    """영화 추천 정보를 나타내는 스키마
    
    제목, 감독, 개봉년도, 장르, 평점을 구조화합니다.
    """

    title: str = Field(description="Movie title")  # 영화 제목
    director: str = Field(description="Director name")  # 감독명
    year: int = Field(description="Release year")  # 개봉년도
    genre: Literal["action", "comedy", "drama", "horror", "sci-fi"] = Field(
        description="Movie genre"
    )  # 장르
    rating: int = Field(description="Rating from 1-5", ge=1, le=5)  # 평점 (1-5)


# 에이전트 생성 - Union 타입으로 여러 추천 유형 지원
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=Union[BookRecommendation, MovieRecommendation], handle_errors=True
    ),
    system_prompt="You are a helpful entertainment recommendation assistant.",
)

# 책 추천 요청
result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Recommend a good science fiction book"}]}
)
print("Book recommendation:")
print(result1["structured_response"])

# 영화 추천 요청
result2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Recommend a comedy movie from the 2000s"}
        ]
    }
)
print("\nMovie recommendation:")
print(result2["structured_response"])

Book recommendation:
title='Dune' author='Frank Herbert' genre='science' rating=5 summary='Set in the distant future, Dune follows Paul Atreides as his family takes control of the desert planet Arrakis, the only source of the valuable spice melange. The story weaves together themes of politics, religion, ecology, and destiny as Paul navigates betrayal, finds his place among the native Fremen, and potentially fulfills an ancient prophecy. This epic tale explores power, survival, and human potential in a richly imagined universe.'

Movie recommendation:
title='Superbad' director='Greg Mottola' year=2007 genre='comedy' rating=4


---

## 정리

이 튜토리얼에서는 LangGraph 에이전트의 구조화된 출력 기능을 학습했습니다. 구조화된 출력을 사용하면 에이전트의 응답을 예측 가능한 형식으로 받아 애플리케이션에서 쉽게 처리할 수 있습니다.

**핵심 개념 요약:**

| 개념 | 설명 |
|:---|:---|
| **ProviderStrategy** | OpenAI, Anthropic, Grok 등 네이티브 지원 모델용 (가장 신뢰성 높음) |
| **ToolStrategy** | 도구 호출을 통한 구조화된 출력 (대부분의 모델 지원) |
| **Union 타입** | 여러 스키마 중 자동 선택 |
| **handle_errors** | 오류 발생 시 자동 재시도 및 수정 |

**스키마 정의 방법:**
- **Pydantic 모델**: 가장 풍부한 검증 기능 제공 (권장)
- **데이터클래스**: 간단한 스키마 정의, 추가 의존성 없음
- **TypedDict**: 딕셔너리 형태로 반환, JSON 직렬화 용이

**실전 팁:**
- 필드에 명확한 `description` 제공
- `Literal` 타입으로 허용값 제한
- `ge`, `le` 등 검증자로 범위 지정
- `Union` 타입으로 다양한 응답 유형 처리
- 프로덕션에서는 `handle_errors=True` 사용 (기본값)

In [7]:
import hashlib
import json
from datetime import datetime
from pathlib import Path

from deepagents import create_deep_agent
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

# 저장 경로: Day01/sample_data/tavily/
# 노트북은 자기 폴더에서 실행되므로, 현재 위치에서 위로 올라가며 Day01 폴더를 찾는다.
DAY01_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Day01")
SAVE_DIR = DAY01_DIR / "sample_data" / "tavily"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# TODO 1: Tavily 검색 도구를 그대로 붙인다 (최대 3개 결과)
tavily = TavilySearch(max_results=3)


# TODO 2: 조회 결과를 저장하는 "도구"를 따로 만들어 에이전트가 직접 호출하게 한다.
# Tavily 도구는 감싸지 않고 그대로 두고, 저장은 별도 도구 호출로 분리했다.
@tool
def save_tavily_result(query: str, results: list[dict]) -> str:
    """Tavily 검색 결과를 Day01/sample_data/tavily/ 에 JSON 파일로 저장한다.

    Tavily 검색을 한 직후에 호출한다.

    Args:
        query: 방금 Tavily에 보낸 검색어
        results: Tavily가 돌려준 results 목록 (title, url, content 등을 그대로)
    """
    # 파일명: 시각 + 검색어 해시 (한글 검색어도 안전하게)
    digest = hashlib.sha1(query.encode("utf-8")).hexdigest()[:8]
    path = SAVE_DIR / f"{datetime.now():%Y%m%d-%H%M%S}-{digest}.json"
    payload = {"query": query, "saved_at": datetime.now().isoformat(), "results": results}
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return f"저장 완료: {path.relative_to(DAY01_DIR.parent)}"


agent = create_deep_agent(
    model=model,
    tools=[tavily, save_tavily_result],
    system_prompt=(
        "웹 검색이 필요하면 Tavily 검색 도구를 사용한다. "
        "Tavily 검색을 할 때마다 바로 이어서 save_tavily_result 도구를 호출해 "
        "검색어와 results를 그대로 저장한 뒤 답변한다."
    ),
)

# 기본 도구 + Tavily 도구 + 저장 도구가 몇 개인지 물어본다.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "너 도구 몇개 있어?"
            }
        ]
    }
)
print(result["messages"][-1].content)

# 실제 검색을 시켜서 검색 → 저장 도구 호출이 이어지는지 확인한다.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "LangChain 최신 버전이 뭔지 웹에서 검색해서 한 줄로 알려줘.",
            }
        ]
    }
)
print(result["messages"][-1].content)

# 호출된 도구 순서와 저장된 파일 확인
print([tc["name"] for m in result["messages"] for tc in getattr(m, "tool_calls", [])])
for f in sorted(SAVE_DIR.glob("*.json")):
    print(f.relative_to(DAY01_DIR.parent), f.stat().st_size, "bytes")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae44-75c7-a78d-d8e2946a99e8; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae45-73e1-894d-3e662f64e017; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae45-73e1-894d-3e662f64e017; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae45-73e1-894d-3e67600f452b; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae45-7606-8ad4-a6d4bc7b9ba6; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae46-744d-a05c-427cfc1118fc; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae46-7311-abb1-542b32877e28; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae46-7264-ac92-97805df371ce; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id

도구는 총 11개 있습니다.

**파일 시스템 (7개)**
1. `ls`: 디렉터리 목록 조회
2. `read_file`: 파일 읽기
3. `write_file`: 파일 생성 또는 덮어쓰기
4. `edit_file`: 문자열 치환 방식으로 파일 수정
5. `delete`: 파일이나 디렉터리 삭제
6. `glob`: 패턴으로 파일 찾기
7. `grep`: 파일 안의 텍스트를 글자 그대로 검색

**작업 위임 (1개)**
8. `task`: 복잡한 작업을 서브에이전트에게 맡기기

**웹 검색 (2개)**
9. `tavily_search`: 웹 검색
10. `save_tavily_result`: Tavily 검색 결과를 `Day01/sample_data/tavily/`에 JSON으로 저장

Tavily로 검색하면 바로 이어서 `save_tavily_result`로 결과를 저장하도록 설정되어 있습니다.

위 목록은 10개이고, 11개는 제가 잘못 센 것입니다. 정확한 개수는 **10개**입니다.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae4b-7252-be7b-48ae10f0f0d9; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae4a-7801-ae86-478861ced8c9; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae4a-7197-b90a-734a2b20ce53; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae46-7264-ac92-97805df371ce; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae46-7311-abb1-542b32877e28; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae46-744d-a05c-427cfc1118fc; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae45-7606-8ad4-a6d4bc7b9ba6; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id=01a1a01e-ae45-73e1-894d-3e67600f452b; trace=01a1a01e-ae44-75c7-a78d-d8e2946a99e8,id

LangChain은 1.x 계열이 최신입니다. Releasebot 기준으로 확인된 가장 최근 `langchain-core` 릴리스는 2026년 10월 23일의 1.7.0입니다. 이 값은 `langchain` 본체가 아니라 core 패키지의 버전입니다. 검색 결과에서 `langchain` 본체의 최신 버전은 확인하지 못했습니다.
['tavily_search', 'save_tavily_result']
Day01/sample_data/tavily/20261103-153458-368636d6.json 3212 bytes
Day01/sample_data/tavily/20261103-153542-368636d6.json 3510 bytes


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f5-7066-9960-d98795dbe457; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f5-7095-b42c-ccc57a7c57fa; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f5-713e-9aa5-8331b330cd03; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f4-7615-a025-24cbe25450a1; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f4-7d05-ab21-f26a53c0de3a; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f4-7475-a52f-2d99b52955d5; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f3-701d-ab9a-d589beaca418; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id=01a1a01e-e6f3-76e6-ac6f-28e8798d1f73; trace=01a1a01e-bc70-70f5-9c80-36aafe98b31a,id

In [ ]:
# TODO: Day01/ 에 있는 정보를 읽어와서,
# 어떤 내용을 가르치는 자료인지 설명하도록 하는 DeepAgents 를 구현.

"""
1. Day01 을 읽도록 만든다.
2. Day01 의 내용을 요약하고, 어떤 내용을 가르치는 자료인지 설명하도록 한다.
3. 요약한 내용을 다시 Day01 폴더에 저장해둔다.

- 단, 코드 개발 시 Tavily 외 새로운 도구를 만들 수 없다.
"""

# 구현 방법: 도구를 새로 만들지 않는다 (tools=[]).
# Deep Agents 의 기본 파일 도구(ls, read_file, glob, grep, write_file)를 그대로 쓰고,
# 이 도구들이 읽고 쓰는 저장소(backend)를 실제 Day01 폴더로 바꾼다.
# 기본 backend(StateBackend)는 메모리에만 있어서 write_file 로 쓴 파일이 디스크에 남지 않는다.
# 대신 Day01 전체가 열리므로, 무엇을 읽고 쓸 수 있는지는 permissions 로 제한한다.

from pathlib import Path

from deepagents import FilesystemPermission, create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend

# 노트북은 자기 폴더에서 실행되므로, 현재 위치에서 위로 올라가며 Day01 폴더를 찾는다.
DAY01_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Day01")

# 에이전트가 보는 경로는 Day01 기준이다. 예: "/README.md" = Day01/README.md
SUMMARY_PATH = "/sample_data/summary/day01_overview.md"

# TODO 1 (읽기): 기본 파일 도구가 실제 Day01 폴더를 보도록 backend 를 바꾼다.
# - default: 실제 Day01 폴더. virtual_mode=True 라 "/" 가 Day01 이고 그 밖으로 나갈 수 없다.
# - "/_agent_scratch/": 에이전트 내부 산출물(하위 에이전트의 큰 결과, 대화 요약)은 메모리에만 둔다.
#   이 route 가 없으면 내부 산출물이 Day01/large_tool_results/ 같은 곳에 실제 파일로 생긴다.
#   이 내부 기록은 아래 permissions 의 쓰기 금지로도 막히지 않는다.
backend = CompositeBackend(
    default=FilesystemBackend(root_dir=DAY01_DIR, virtual_mode=True),
    routes={"/_agent_scratch/": StateBackend()},
    artifacts_root="/_agent_scratch",
)

# 권한: 위에서부터 처음 맞는 규칙 하나만 적용되고, 맞는 규칙이 없으면 허용이다.
# ("/**" 는 .env 같은 점(.) 파일에 맞지 않으므로 점 파일 규칙을 따로 둔다)
permissions = [
    # .env 같은 점 파일·점 폴더는 읽기와 쓰기 모두 금지 (API 키 보호)
    FilesystemPermission(operations=["read", "write"], paths=["/**/.*", "/**/.*/**"], mode="deny"),
    # 그림·PDF·DB 는 읽지 않는다 (PDF·그림은 모델에 파일째 전달되어 비용이 크다)
    FilesystemPermission(operations=["read"], paths=["/**/*.{png,jpg,jpeg,gif,webp,avif,pdf,db}"], mode="deny"),
    # 이전 실습 산출물과 이 셀이 저장한 요약은 요약 대상이 아니다
    FilesystemPermission(operations=["read"], paths=["/sample_data/**"], mode="deny"),
    # TODO 3 (저장): 쓰기는 요약 저장 폴더와 메모리 작업 공간에만 허용한다
    FilesystemPermission(operations=["write"], paths=["/sample_data/summary/**", "/_agent_scratch/**"], mode="allow"),
    # 나머지 Day01 은 쓰기·수정·삭제 모두 금지 (강의 노트북 보호)
    FilesystemPermission(operations=["write"], paths=["/**"], mode="deny"),
]

# TODO 2 (요약): 무엇을 읽고, 무엇을 써서, 어디에 저장할지 지시한다.
system_prompt = f"""너는 강의 자료를 분석하는 조교다.
파일 시스템의 "/" 는 Day01 강의 자료 폴더다. 
Day01 이 무엇을 가르치는 자료인지 파악해서 요약 문서를 저장한다.

진행 순서
1. /README.md 와 /UPSTREAM.md 를 먼저 읽는다. README 에 PART 별 튜토리얼 목록이 있다.
2. glob 으로 실제 노트북 목록("**/*.ipynb")을 확인하고 README 목록과 대조한다. 둘이 다르면 실제 파일을 따른다.
3. README 설명이 짧은 부분(특히 /Appendix)만 노트북을 read_file 로 열어 앞부분(limit=40)을 읽는다.
   노트북 전체를 읽지 않는다. 모든 노트북을 열 필요도 없다.
4. 요약을 마크다운으로 써서 write_file 로 {SUMMARY_PATH} 에 저장한다. 다음 내용을 넣는다.
   - 이 자료가 무엇을 가르치는가 (학습 목표, 다루는 라이브러리)
   - PART 01~05 와 Appendix 별 핵심 내용과 대표 노트북 경로
   - 권장 학습 순서
   - 근거로 읽은 파일 목록
5. 저장한 경로와 세 줄 요약으로 답한다.

규칙
- 권한이 거부된 파일(.env, 그림, PDF 등)은 다시 시도하지 않는다.
- 실제로 있는 파일 이름만 적는다.
- 파일은 {SUMMARY_PATH} 한 곳에만 쓴다.
"""

agent = create_deep_agent(
    model=model,  # 위에서 만든 model 을 그대로 쓴다
    tools=[],  # 새 도구 없음: 기본 파일 도구만 쓴다
    middleware=[],
    system_prompt=system_prompt,
    backend=backend,
    permissions=permissions,
)

# recursion_limit: Deep Agents 기본값은 9,999 라 사실상 상한이 없다. 비용 보호용으로 직접 정한다.
# 도구 호출 한 번에 그래프 2단계, 시작과 끝에 3단계가 든다. 80 이면 모델 호출은 최대 39회다.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Day01 폴더를 읽고 어떤 내용을 가르치는 자료인지 요약해서 저장해줘.",
            }
        ]
    },
    config={"recursion_limit": 80},
)
print(result["messages"][-1].content)

# 확인 1: 호출된 도구와 대상 경로
for m in result["messages"]:
    for tc in getattr(m, "tool_calls", []):
        args = tc["args"]
        print(tc["name"], args.get("file_path") or args.get("path") or args.get("pattern") or "")

# 확인 2: 권한 등으로 실패한 도구 호출
for m in result["messages"]:
    if getattr(m, "status", None) == "error":
        print("거부/실패:", m.name, str(m.content)[:100])

# 확인 3: 실제 디스크에 저장된 요약 파일
saved = DAY01_DIR / SUMMARY_PATH.lstrip("/")
print(saved.relative_to(DAY01_DIR.parent), saved.stat().st_size, "bytes")

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e3-7365-a90f-067191907c81; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e4-724b-bb8a-875d6c866189; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e4-724b-bb8a-875d6c866189; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e4-724b-bb8a-875efe9573a3; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e4-704e-aab2-125c24d319e5; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e4-76b9-8ca5-c6c519dc4812; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e5-7d88-ac75-eb24a7df9e4c; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a043-61e5-72fc-8631-b69487c35828; trace=01a1a043-61e3-7365-a90f-067191907c81,id

요약을 `/sample_data/summary/day01_overview.md`에 저장했습니다.

- **내용**: Day01은 LangGraph V1과 LangChain 1.x로 AI 에이전트를 만드는 한국어 Jupyter 튜토리얼입니다. 그래프 기초(PART01)에서 시작해 에이전트(PART02), 미들웨어·메모리·MCP 확장(PART03), 멀티에이전트(PART04), 서브그래프·스트리밍 같은 핵심 기능(PART05)까지 이어집니다.
- **Appendix**: A는 RAG 응용(Naive RAG에서 Agentic·Corrective·Self·Adaptive RAG까지), B는 Plan-and-Execute·SQL Agent 같은 유스케이스 5개, C는 Neo4j Text2Cypher(GraphRAG)입니다. A에는 README에 없는 노트북이 2개 있습니다(`01-LangGraph-Building-Graphs`, `10-LangGraph-Retrieval`).
- **학습 순서**: 환경을 설정한 뒤 PART01부터 PART05, Appendix 순으로 진행합니다.

README와 실제 노트북(51개)을 대조했을 때 PART01~05 목록은 일치했습니다. README는 챕터 폴더 이름을 줄여 적었기 때문에, 요약에는 실제 한글 폴더 경로를 적었습니다. Appendix는 대표 노트북 5개의 앞부분(40줄)만 읽었습니다. PART01~05 노트북은 열지 않았고, 그 내용은 README 설명을 따랐습니다.
read_file /README.md
read_file /UPSTREAM.md
glob **/*.ipynb
read_file /README.md
read_file /Appendix/A-RAG응용/01-LangGraph-Building-Graphs.ipynb
read_file /Appendix/A-RAG응용/10-LangGraph-Retrieval.ipynb
read_file /Appendix/A-RAG응용/02-LangGraph-Naive-RAG.ipynb

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005f-71f8-8085-3eb74a768898; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005e-7426-b1df-c01949b65d1b; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005e-7032-b4e3-4100e7fe11ef; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005c-7b7c-84cd-493f90251729; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005c-7d16-9fc6-64af9d1e34b3; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005c-7e39-910e-722e6934df8f; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005b-7787-aa1b-17e072b9679a; trace=01a1a043-61e3-7365-a90f-067191907c81,id=01a1a044-005b-73e5-b01a-4f8aab8b4e8b; trace=01a1a043-61e3-7365-a90f-067191907c81,id

In [11]:
result["messages"]

[HumanMessage(content='Day01 폴더를 읽고 어떤 내용을 가르치는 자료인지 요약해서 저장해줘.', additional_kwargs={}, response_metadata={}, id='7848c1a9-4f74-4afa-bd06-9e62a5c00f98'),
 AIMessage(content=[{'id': 'toolu_014PZBydoE7Ao8ADxFz3cPAJ', 'caller': {'type': 'direct'}, 'input': {'file_path': '/README.md', 'limit': 200}, 'name': 'read_file', 'type': 'tool_use', 'toolset_name': None}, {'id': 'toolu_016iC6GGJz1HRF2oWjkYvoHz', 'caller': {'type': 'direct'}, 'input': {'file_path': '/UPSTREAM.md'}, 'name': 'read_file', 'type': 'tool_use', 'toolset_name': None}, {'id': 'toolu_017zsUrRQ7vWH9PFnz9s58YB', 'caller': {'type': 'direct'}, 'input': {'pattern': '**/*.ipynb'}, 'name': 'glob', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CgfankKK2f8ozGPa2jHVs', 'container': None, 'diagnostics': None, 'model': 'claude-sonnet-5-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_i

In [ ]:
# TODO 3: .agents/skills 폴더에 있는 스킬을 로드하여, 
# PI Agent 의 내용을 웹에서 찾고 그것을 DeepAgents 로 구현하기 위해서는 
# 어떻게 코드를 구성하도록 해야하는지 설계 및 코드 구현
